# 02 — BCRD CPI & Family Basket | Bronze Ingestion (incremental, Auto Loader)

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Ingest the six official BCRD price-index workbooks into Bronze **exactly as published**, so that Silver can parse each layout with full lineage back to the source cell.

## Why row-level capture
These workbooks are presentation tables, not datasets: titles, multi-row headers, a year written once per block, Spanish month names, index/variation column pairs, and methodology footnotes. Each file has a different layout. Forcing them into typed columns at ingestion would mean making interpretation decisions in Bronze.

Bronze therefore stores **every row of every sheet** as an array of text cells, plus lineage metadata. Nothing is interpreted, dropped or corrected here — including the methodology notes, which Silver keeps as documented context.

## Sources (raw archive `/Volumes/rd_economic_intelligence/bronze/bcrd_raw/cpi/<yyyy>/<mm>/`, one folder per arrival month)
| dataset_code | File | Content |
|---|---|---|
| `cpi_national` | `ipc_base_2019-2020.xlsx` | National CPI index and variations, 1984 onward |
| `cpi_groups` | `ipc_grupos_base_2019-2020.xlsx` | CPI by 12 expenditure groups, 1999 onward |
| `cpi_core` | `ipc_subyacente_base_2019-2020.xlsx` | Core CPI, 2000 onward |
| `cpi_regions` | `ipc_regiones_base_2019-2020.xlsx` | CPI by region, 2011 onward |
| `basket_quintiles` | `Costo_Canasta_quintiles_base_2019-2020.xlsx` | Family basket cost by income quintile and national, 2018 onward |
| `basket_regions` | `Costo_Canasta_regiones_base_2019-2020.xlsx` | Family basket cost by region, Oct-2020 onward |

**File register:** `bronze.bcrd_cpi_files` — **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`) records each new workbook once; the file name identifies the dataset (an unknown name fails the run).

**Target:** `rd_economic_intelligence.bronze.bcrd_cpi_raw_rows` — one snapshot per archived workbook. Each workbook carries the full history, so Silver uses the newest snapshot of each dataset.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
# Step 0 — Dependencies: serverless does not ship openpyxl, required to read .xlsx
dbutils.library.restartPython()

In [0]:
# ============================================================
# Step 1 — Configuration and Auto Loader file discovery
# ============================================================
from datetime import datetime, timezone
import sys, uuid, json, importlib
import openpyxl
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/bcrd_raw/cpi/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/bcrd_raw/_checkpoints/cpi_files/"
FILES_TABLE = f"{CATALOG}.bronze.bcrd_cpi_files"
TARGET_TABLE = f"{CATALOG}.bronze.bcrd_cpi_raw_rows"

# file name -> (dataset_code, text that must appear in the header area: structural contract)
SOURCES = {
    "ipc_base_2019-2020.xlsx":                     ("cpi_national",     ["Período", "Indice"]),
    "ipc_grupos_base_2019-2020.xlsx":              ("cpi_groups",       ["Período", "Alimentos y Bebidas"]),
    "ipc_subyacente_base_2019-2020.xlsx":          ("cpi_core",         ["Período", "Subyacente"]),
    "ipc_regiones_base_2019-2020.xlsx":            ("cpi_regions",      ["Período", "Región Ozama"]),
    "Costo_Canasta_quintiles_base_2019-2020.xlsx": ("basket_quintiles", ["Período", "Quintil 1", "Nacional"]),
    "Costo_Canasta_regiones_base_2019-2020.xlsx":  ("basket_regions",   ["Período", "Región Ozama"]),
}
HEADER_SCAN_ROWS = 10
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, RAW_PATH, "*.xlsx", CHECKPOINT, FILES_TABLE,
    comment="Bronze: register of BCRD CPI and family-basket workbooks found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES_TABLE, TARGET_TABLE)
unknown = [f["_source_file"] for f in pending if f["_source_file"] not in SOURCES]
if unknown:
    raise ValueError(f"Unrecognised workbook(s) in the archive (new base year or renamed file?): {unknown}")
print(f"Batch ID: {BATCH_ID} · Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'] for f in pending]}")

In [0]:
# ============================================================
# Step 2 — Per pending file: every row of every sheet as text, structural contract, append
# ============================================================
def as_text(value):
    """Text exactly as stored: floats keep full precision (repr), empty cells stay None."""
    if value is None:
        return None
    if isinstance(value, float):
        return repr(value)
    return str(value)

schema = ("dataset_code string, _source_sheet string, _source_sheet_index int, "
          "_source_row_number int, cells array<string>, non_empty_cells int")
captured = {}
for f in pending:
    dataset_code, required_tokens = SOURCES[f["_source_file"]]
    wb = openpyxl.load_workbook(ING.local_path(f["_source_path"]), read_only=True, data_only=True)
    rows, header_text = [], ""
    for sheet_index, ws in enumerate(wb.worksheets):
        for row_number, values in enumerate(ws.iter_rows(values_only=True), start=1):
            cells = [as_text(v) for v in values]
            while cells and cells[-1] is None:
                cells.pop()
            if sheet_index == 0 and row_number <= HEADER_SCAN_ROWS:
                header_text += " ".join(c for c in cells if c) + " "
            rows.append((dataset_code, ws.title, sheet_index, row_number, cells, sum(1 for c in cells if c is not None)))
    wb.close()
    absent = [t for t in required_tokens if t not in header_text]
    if absent:
        raise ValueError(f"{f['_source_path']}: source layout changed (missing header tokens {absent})")
    ING.lineage(spark.createDataFrame(rows, schema), f, BATCH_ID, INGESTION_TS).write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
    captured[f["_source_path"].split("/cpi/")[-1]] = {"dataset_code": dataset_code, "rows": len(rows)}
    print(f"{dataset_code:<17} {f['_source_file']:<45} rows {len(rows):,} · contract PASS")

if spark.catalog.tableExists(TARGET_TABLE):
    spark.sql(f"""COMMENT ON TABLE {TARGET_TABLE} IS
        'Bronze: BCRD CPI and family-basket workbooks captured row by row as published (text cells), one snapshot per archived workbook, with lineage. Parsed into tidy tables in Silver.'""")

In [0]:
snapshots = spark.table(TARGET_TABLE).select("_source_path").distinct().count() if spark.catalog.tableExists(TARGET_TABLE) else 0
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
                                  "captured": captured, "snapshots_in_bronze": snapshots}, ensure_ascii=False))